In [1]:
import os
import io
import json
import time
from typing import List, Dict, Any
import fitz
from PIL import Image
from dotenv import load_dotenv
from langchain.schema import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI


from pylatex import Document, Section, Subsection, Command, Package
from pylatex.utils import NoEscape, bold
from pylatex.base_classes import Environment
from datetime import datetime

In [3]:
import re

def filter_summary(text: str) -> str:
    """
    Filters out irrelevant or filler sentences from the provided text.

    Args:
        text (str): The original text containing potential irrelevant content.

    Returns:
        str: The filtered text with irrelevant sentences removed.
    """
    # Define patterns to exclude irrelevant sentences
    exclusion_patterns = [
        r"\b(In summary|Please note|Important Note|Unfortunately)\b",  # Common filler phrases
        r"\b(comprehensive overview|slides discuss|examples provided|difficult to extract)\b",  # Generic terms
        r"Without proper formatting|poor image quality|accuracy and completeness",  # Irrelevant explanations
        r"\b(jumbled collection|difficult to extract|challenging due to)\b",  # Complaints about image quality
    ]
    
    # Compile the patterns into a regex
    combined_pattern = re.compile("|".join(exclusion_patterns), re.IGNORECASE)
    
    # Filter out sentences matching any exclusion pattern
    filtered_sentences = [
        sentence.strip()
        for sentence in text.split('. ')
        if not combined_pattern.search(sentence)
    ]
    
    # Reassemble filtered sentences
    return ". ".join(filtered_sentences)


In [19]:
class NotesProcessor:
    def __init__(self, 
                 model='gemini-1.5-flash', 
                 temperature=0, 
                 max_tokens=None, 
                 timeout=None, 
                 max_retries=2):
        """
        Initialize the NotesProcessor with Gemini API configuration.
        """
        load_dotenv()  # Load environment variables
        
        self.llm = ChatGoogleGenerativeAI(
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            timeout=timeout,
            max_retries=max_retries
        )
        
        # Predefined category prompts
        self.category_prompts = {
            "Key Terms": "Extract the key terms from the following slides and provide a clear and concise definition for each one.",
            "Problem Types": "Identify the types of problems discussed in the following slides and provide examples if possible.",
            "Formulas": "Extract the key formulas from the following slides, and explain their meaning briefly."
        }
        
        self.previous_terms = ["linear programming"]

    def extract_pdf_content(self, file_path: str) -> tuple:
        """
        Extract text and images from a PDF file.
        """
        pdf_content = ""
        images = []

        with fitz.open(file_path) as pdf:
            for page in pdf:
                pdf_content += page.get_text()

                for img in page.get_images(full=True):
                    xref = img[0]
                    base_image = pdf.extract_image(xref)
                    image_bytes = base_image["image"]
                    image = Image.open(io.BytesIO(image_bytes))
                    images.append(image)

        return pdf_content, images

    def robust_generate(self, message: HumanMessage, retries: int = 5) -> str:
        """
        Robust method for generating text with exponential backoff.
        """
        for attempt in range(retries):
            try:
                response = self.llm.generate([[message]])
                return response.generations[0][0].text
            except Exception as e:
                if "ResourceExhausted" in str(e) and attempt < retries - 1:
                    wait_time = 5 * (attempt + 1)
                    print(f"Resource exhausted. Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    raise RuntimeError(f"Failed after {retries} retries: {e}")

    def process_batch(self, 
                      slides: List[str], 
                      category_prompt: str, 
                      batch_index: int) -> str:
        print(f"Processing batch {batch_index + 1} for: {category_prompt}")
        combined_text = "\n".join(slides)
        message = HumanMessage(content=[
            {"type": "text", "text": category_prompt},
            {"type": "text", "text": "The following terms have already been generated. Do not repeat them: " + ", ".join(self.previous_terms)},
            {"type": "text", "text": combined_text},
        ])
        return self.robust_generate(message)
    
    def clean_result(self, result: str, lecture_name: str) -> str:
        """
        Clean up the result by adding <L[lecture name] page_number> instead of <SLIDE page_number>.
        """
        # find all <SLIDE page_number> and replace with <L[lecture name] page_number>
        result = re.sub(r'<SLIDE (\d+)>', f'<L[{lecture_name}] \\1>', result)
        return result
    
    def prune_term(self, result: str, term: str) -> str:
        """
        Prune the term from the result. All terms will be in the format term: definition, seperated by newlines. 
        """
        return "\n".join([line for line in result.splitlines() if term.lower() not in line.lower()])
    
    def get_terms(self, result: str) -> List[str]:
        """
        Get all terms from the result. All terms will be in the format term: definition, seperated by newlines. All terms will be converted to lowercase to avoid duplicates.
        """
        terms = []
        # remove all <SLIDE page_number>
        result = re.sub(r'<SLIDE \d+>', '', result)
        
        for line in result.splitlines():
            if ":" in line:
                formatted_line = line.split(":")[0].strip().lower().strip("*")
                # remove parentheses
                formatted_line = re.sub(r'\([^)]*\)', '', formatted_line)
                terms.append(formatted_line)
        print("Terms: ", terms)
        return terms
    
    def process_slides_with_categories_updated(self, 
                                            folder_path: str, 
                                            num_slides: int = None,
                                            batch_size: int = 1, 
                                            custom_categories: Dict[str, str] = None) -> Dict[str, List[str]]:
        """
        Process slides, extract content in batches, and generate a single cohesive summary.
        """
        # Generate timestamp for output file
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        output_file = f"jsonsummaries/processed_notes_{timestamp}.json"

        # Use custom categories if provided, otherwise use defaults
        category_prompts = custom_categories or self.category_prompts

        slides = []
        slide_names = []
        for file_name in sorted(os.listdir(folder_path))[:num_slides]:
            if file_name.endswith('.txt'):
                slide_path = os.path.join(folder_path, file_name)
                with open(slide_path, 'r') as file:
                    slides.append(file.read())
                    slide_names.append(file_name.split('.')[0])
        
        # Process each category and aggregate results
        categorized_results = {category: [] for category in category_prompts.keys()}
        for i in range(0, len(slides), batch_size):
            batch = slides[i:i + batch_size]
            for category, prompt in category_prompts.items():
                try:
                    result = self.process_batch(batch, prompt, i // batch_size)
                    cleaned_result = self.clean_result(result, slide_names[i])
                    new_terms = self.get_terms(result)
                    for term in new_terms:
                        if term in self.previous_terms:
                            print(f"Pruning term: {term}")
                            cleaned_result = self.prune_term(cleaned_result, term)
                        else:
                            self.previous_terms.append(term)
                    categorized_results[category].append(cleaned_result)
                except Exception as e:
                    print(f"Error processing batch {i // batch_size} for {category}: {e}")

        # Save results to file
        with open(output_file, "w") as file:
            json.dump(categorized_results, file, indent=4)

        return categorized_results

    def generate_concise_summary(self, 
                                 categorized_results: Dict[str, List[str]]) -> str:
        """
        Generate a concise summary from categorized results.
        """
        final_summary = ""
        for category, results in categorized_results.items():
            combined_results = "\n".join(results)
            summary_prompt = (
                f"You are an expert summarization assistant tasked with creating a comprehensive and cohesive summary of the {category.lower()}. Follow these precise guidelines:\n"
                "1. Synthesize Information:\n"
                f"- Generate a summary that captures the OVERALL essence of the {category.lower()}\n"
                "- Exclude details specific to individual slides or instances\n"
                "- Focus on broad, generalizable concepts and key insights\n\n"
                "2. Formatting Requirements:\n"
                "- Combine term and definition into a SINGLE, concise bullet point\n"
                "- Ensure each bullet point is a complete, informative sentence\n"
                "- Avoid breaking definitions across multiple bullet points\n"
                "- Maintain a clear, flowing narrative that connects key points logically\n\n"
                "3. Content Criteria:\n"
                "- Prioritize the most significant and impactful information\n"
                "- Eliminate redundant or overly specific details\n"
                "- Present information in a way that provides a holistic understanding\n"
                "- Use precise, academic language that conveys depth and nuance\n\n"
                "4. Structure:\n"
                "- Begin with a brief introductory statement defining the core concept\n"
                "- Organize bullet points to create a logical progression of ideas\n"
                "- Ensure each point adds unique value to the overall summary\n\n"
                "5. Final Review:\n"
                "- Check that the summary reads as a cohesive, integrated overview\n"
                "- Verify that no point feels isolated or disconnected from the whole\n"
                "- Confirm that the summary provides a comprehensive yet concise understanding\n\n"
                "Generate the summary strictly adhering to these guidelines. "
                "Keep all HTML links in the slides intact."
            )
            summary_message = HumanMessage(content=[
                {"type": "text", "text": summary_prompt},
                {"type": "text", "text": combined_results},
            ])
            final_summary_response = self.robust_generate(summary_message)
            final_summary += f"\n--- {category} ---\n{final_summary_response}\n"
            
        # save to markdown file
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        with open(f"markdownsummaries/Summary_{timestamp}.md", "w", encoding="utf-8") as file:
            file.write(final_summary)
        return final_summary

    def save_results(self, 
                    categorized_results: Dict[str, List[str]]):
        """
        Save processed results to JSON and Markdown files with improved formatting.
        
        Args:
            categorized_results (dict): Processed results by category
        """
        # Generate timestamp for filenames
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        
        # Set default filenames with timestamp if not provided
        json_file = f"jsonsummaries/processed_notes_{timestamp}.json"
        markdown_file = f"markdownsummaries/Summary_{timestamp}.md"
        
        # Save JSON
        with open(json_file, "w") as file:
            json.dump(categorized_results, file, indent=4)
        
        # Save Markdown with improved formatting
        with open(markdown_file, "w", encoding="utf-8") as file:
            for category, results in categorized_results.items():
                # Use heading for category
                file.write(f"# {category}\n\n")
                
                for idx, result in enumerate(results, 1):
                    # Create subheadings for batches
                    file.write(f"## {category} Summary {idx}\n\n")
                    
                    # Format the result
                    cleaned_result = result.strip()
                    formatted_result = "\n".join(f"- {line.strip()}" for line in cleaned_result.splitlines() if line.strip())
                    
                    file.write(f"{formatted_result}\n\n")
                
                file.write("\n---\n")

            file.write("*Generated by NotesProcessor*\n")


    def save_results_html(self, 
                          categorized_results: Dict[str, List[str]]):
        """
        Save processed results to an HTML file, filtering out irrelevant sentences.

        Args:
            categorized_results (dict): Processed results by category
            html_file (str): Path to save the HTML summary
        """
        # Generate timestamp for filename
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        html_file = f"htmlsummaries/Summary_{timestamp}.html"
        
        with open(html_file, "w", encoding="utf-8") as file:
            # Start HTML structure
            file.write("""
            <!DOCTYPE html>
            <html lang="en">
            <head>
                <meta charset="UTF-8">
                <meta name="viewport" content="width=device-width, initial-scale=1.0">
                <title>Summary</title>
                <style>
                    body {
                        font-family: Arial, sans-serif;
                        line-height: 1.6;
                        margin: 20px;
                        background-color: #f9f9f9;
                        color: #333;
                    }
                    .container {
                        max-width: 800px;
                        margin: 0 auto;
                        background: #fff;
                        padding: 20px;
                        border-radius: 8px;
                        box-shadow: 0 0 10px rgba(0, 0, 0, 0.1);
                    }
                    h1 {
                        text-align: center;
                        color: #444;
                    }
                    h2 {
                        border-bottom: 2px solid #ddd;
                        padding-bottom: 5px;
                        color: #555;
                        page-break-before: always;
                    }
                    ul {
                        padding-left: 20px;
                    }
                    li {
                        margin: 5px 0;
                    }
                    .footer {
                        text-align: center;
                        margin-top: 20px;
                        font-size: 0.9em;
                        color: #777;
                    }
                </style>
            </head>
            <body>
                <div class="container">
                    <h1>Summary</h1>
            """)

            def has_valid_key_term(line):
                # Check if line contains a colon with text on both sides
                parts = line.split(':')
                if len(parts) != 2:
                    return False
                return bool(parts[0].strip()) and bool(parts[1].strip())

            def extract_slide_fragment(line):
                """
                Extract slide information from citations in the format <L[lecture_name] page_number>
                Returns lecture_name.pdf#page=page_number
                """
                # Updated regex pattern to capture both the lecture name and page number
                matches = re.findall(r'<L\[([^\]]+)\] (\d+)>', line)
                if matches:
                    # matches will be a list of tuples: [(lecture_name, page_number), ...]
                    lecture_name, page_number = matches[0]  # Get first match
                    return f"{lecture_name}.pdf#page={page_number}"
                return None

            # Write categorized results
            for category, results in categorized_results.items():
                file.write(f"<h2>{category}</h2>\n<ul>\n")
                for result in results:
                    result = result.strip().replace("*", "")
                    cleaned_result = filter_summary(result.strip())
                    if cleaned_result:
                        formatted_result = []
                        for line in cleaned_result.splitlines():
                            line = line.strip()
                            if line and has_valid_key_term(line):
                                # Extract term and description
                                term, description = line.split(':', 1)  # Use split with maxsplit=1
                                term = term.strip()
                                description = description.strip()
                                slide_fragment = extract_slide_fragment(line)
                                
                                if slide_fragment:
                                    # Create the full line with linked term
                                    line = f"<li><a href='https://www.math.purdue.edu/~yipn/421/{slide_fragment}'>{term}</a>: {description}"
                                else:
                                    line = f"<li>{term}: {description}"

                                # Remove slide markers
                                line = re.sub(r'<L\[[^>]+] [^>]+>', '', line)
                                # Add closing li tag
                                line += "</li>"
                                formatted_result.append(line)
                        
                        if formatted_result:
                            file.write(f"{''.join(formatted_result)}\n")
                file.write("</ul>\n")
            

    def save_results_latex(self, categorized_results: Dict[str, List[str]]):
        """
        Save processed results to a LaTeX PDF file using PyLaTeX.
        
        Args:
            categorized_results (dict): Processed results by category
        """
        # Create document with custom geometry and packages
        geometry_options = {
            "margin": "1in",
            "headheight": "14pt",
            "headsep": "25pt"
        }
        doc = Document(geometry_options=geometry_options)
        
        # Add required packages
        doc.packages.append(Package('hyperref'))
        doc.packages.append(Package('enumitem'))
        doc.packages.append(Package('fancyhdr'))
        doc.packages.append(Package('xcolor'))
        
        # In your preamble configuration
        doc.preamble.append(NoEscape(r'''
            \hypersetup{
                colorlinks=true,
                linkcolor=blue,
                filecolor=magenta,
                urlcolor=blue,
            }
            
            \pagestyle{fancy}
            \fancyhf{}
            \rhead{Generated on \today}
            \lhead{Course Summary}
            \cfoot{\thepage}
            
            % Configure itemize settings globally
            \setlist[itemize]{nosep}  % Reduce vertical spacing between items
            \setlist[itemize]{leftmargin=*}  % Align with left margin
        '''))
        
        # Add title
        doc.preamble.append(Command('title', 'Linear Programming Course Summary'))
        doc.preamble.append(Command('author', 'Generated by Scribe.AI'))
        doc.preamble.append(Command('date', NoEscape(r'\today')))
        
        # Begin document
        doc.append(NoEscape(r'\maketitle'))
        
        def has_valid_key_term(line):
            parts = line.split(':')
            return len(parts) == 2 and bool(parts[0].strip()) and bool(parts[1].strip())
        
        def extract_slide_fragment(line):
            """
            Extract slide information from citations in the format <L[lecture_name] page_number>
            Returns lecture_name.pdf#page=page_number
            """
            # Updated regex pattern to capture both the lecture name and page number
            matches = re.findall(r'<L\[([^\]]+)\] (\d+)>', line)
            if matches:
                # matches will be a list of tuples: [(lecture_name, page_number), ...]
                lecture_name, page_number = matches[0]  # Get first match
                return f"{lecture_name}.pdf#page={page_number}"
            return None
        
        # Update the CustomItemize class
        class CustomItemize(Environment):
            _latex_name = 'itemize'
            
            def __init__(self):
                super().__init__()
                self.options = NoEscape('leftmargin=*')
        
        # Process each category
        for category, results in categorized_results.items():
            with doc.create(Section(category)):
                # Create one itemize environment for all items in this category
                with doc.create(CustomItemize()):
                    for result in results:
                        result = result.strip().replace("*", "")
                        cleaned_result = filter_summary(result.strip())
                        
                        if cleaned_result:
                            for line in cleaned_result.splitlines():
                                line = line.strip()
                                if line and has_valid_key_term(line):
                                    slide_fragment = extract_slide_fragment(line)
                                    
                                    # Remove slide markers
                                    line = re.sub(r'<L\[[^>]+] [^>]+>', '', line)
                                    
                                    term, description = line.split(':', 1)
                                    term = term.strip()
                                    description = description.strip()
                                    
                                    if slide_fragment:
                                        # Create item with hyperlink
                                        item_content = NoEscape(
                                            f"\\item \\href{{https://www.math.purdue.edu/~yipn/421/{slide_fragment}}}{{{bold(term)}}}: {description}"
                                        )
                                    else:
                                        item_content = NoEscape(f"\\item {bold(term)}: {description}")
                                    
                                    doc.append(item_content)
                
                # Add page break after each section
                # doc.append(NoEscape(r'\newpage'))
        
        # Generate timestamp for filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"latexsummaries/Summary_{timestamp}"
        
        try:
            # Generate PDF
            doc.generate_pdf(filename, clean_tex=False)
            print(f"PDF generated successfully: {filename}.pdf")
            
            # Optionally clean up the .tex file
            if os.path.exists(f"{filename}.tex"):
                os.remove(f"{filename}.tex")
                
        except Exception as e:
            print(f"Error generating PDF: {str(e)}")
            # Keep the .tex file for debugging if generation fails
            if os.path.exists(f"{filename}.tex"):
                print(f"LaTeX source file preserved for debugging: {filename}.tex")



In [18]:
def main():
    """
    Example usage of the NotesProcessor
    """
    processor = NotesProcessor()
    
    # Custom categories example
    custom_categories = {
        "Key Terms": "Extract the key terms from the following slides and provide a clear and concise definition for each one. Your key terms should be specific to this lecture, but also make sense as a general topic in the context of Linear Programming. Respond in the following format: <term>: <definition>. Do not include any other text, like numbering, intermediate references, or general summaries before/after the term. Do not focus on generating Problem Types or Algorithm Solutions since this will be done in another section. If you are citing a slide, include the slide number at the end of the term. Here is an example: 'Normal Equation: A closed-form solution for the least squares problem in linear regression. It's always solvable, even if the original system of equations is not.<SLIDE 12>'. If citing multiple slides, include the slide numbers at the end of the definition. Here is another example: 'Support Vectors: The data points closest to the hyperplane in an SVM. They are the most influential points in determining the hyperplane.<SLIDE 10><SLIDE 12><SLIDE 17>'.",
        
        "Problem Types": "Extract the key types of problems discussed in the following slides and provide examples if possible. Your problem types should be specific to this lecture, but also make sense as a general problem in the context of Linear Programming. Respond in the following format: <problem type>: <description>. Do not include any other text, like numbering, intermediate references, or general summaries before/after the problem type. Do not focus on generating Key Terms or Algorithm Solutions since this will be done in another section. If you are citing a slide, include the slide number at the end of the term. Here is an example: 'Verifying Optimality: A method for verifying the optimality of a solution is presented, involving checking the objective function value and the feasibility of the dual solution.<SLIDE 13>'. If citing multiple slides, include the slide numbers at the end of the description. Here is another example: 'Determining the existence of a non-negative solution to `Ax = b`: This problem investigates whether there exists a vector `x` with non-negative components that satisfies the equation `Ax = b`. Several equivalent conditions are presented using a vector `y`. <SLIDE 2><SLIDE 3><SLIDE 4><SLIDE 5><SLIDE 6><SLIDE 7><SLIDE 8><SLIDE 9><SLIDE 10><SLIDE 11>'.",
        
        "Algorithm Solutions": "Extract the key algorithms to solve the problems from the following slides, and explain their meaning briefly. Your algorithms should be specific to this lecture, but also make sense as a general algorithm solution in the context of Linear Programming. Using formulas is encouraged. Respond in the following format: <algorithm>: <formula and/or explanation>. Do not include any other text, like numbering, intermediate references, or general summaries before/after the algorithm. Do not focus on generating Key Terms or Problem Types since this will be done in another section. If you are citing a slide, include the slide number at the end of the term. Here is an example: 'Strong Duality: This theorem states that the optimal objective function values of the primal and dual problems are equal.<SLIDE 2>'. If citing multiple slides, include the slide numbers at the end of the term. Here is another example: 'Caratheodory's Theorem: This theorem states that any point in the convex hull of a set in Rm can be expressed as a convex combination of at most m+1 points. This significantly reduces the computational complexity of algorithms dealing with convex hulls, as it limits the number of points that need to be considered.<SLIDE 8><SLIDE 9>'."
    }

    folder_path = "output"
    
    # Process slides with default or custom categories
    categorized_results = processor.process_slides_with_categories_updated(
        folder_path, 
        batch_size=1, 
        custom_categories=custom_categories
    )
    
    # Generate summary
    concise_summary = processor.generate_concise_summary(categorized_results)
    print(concise_summary)
    
    # Save results
    processor.save_results_html(categorized_results)
    processor.save_results_latex(categorized_results)

In [20]:
main()

I0000 00:00:1733260219.054003 9035632 check_gcp_environment_no_op.cc:29] ALTS: Platforms other than Linux and Windows are not supported


Processing batch 1 for: Extract the key terms from the following slides and provide a clear and concise definition for each one. Your key terms should be specific to this lecture, but also make sense as a general topic in the context of Linear Programming. Respond in the following format: <term>: <definition>. Do not include any other text, like numbering, intermediate references, or general summaries before/after the term. Do not focus on generating Problem Types or Algorithm Solutions since this will be done in another section. If you are citing a slide, include the slide number at the end of the term. Here is an example: 'Normal Equation: A closed-form solution for the least squares problem in linear regression. It's always solvable, even if the original system of equations is not.<SLIDE 12>'. If citing multiple slides, include the slide numbers at the end of the definition. Here is another example: 'Support Vectors: The data points closest to the hyperplane in an SVM. They are the 

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


Terms:  ['dual of general lp problem', 'dual of general lp problem ', 'weak duality theorem']
Pruning term: weak duality theorem
Processing batch 9 for: Extract the key terms from the following slides and provide a clear and concise definition for each one. Your key terms should be specific to this lecture, but also make sense as a general topic in the context of Linear Programming. Respond in the following format: <term>: <definition>. Do not include any other text, like numbering, intermediate references, or general summaries before/after the term. Do not focus on generating Problem Types or Algorithm Solutions since this will be done in another section. If you are citing a slide, include the slide number at the end of the term. Here is an example: 'Normal Equation: A closed-form solution for the least squares problem in linear regression. It's always solvable, even if the original system of equations is not.<SLIDE 12>'. If citing multiple slides, include the slide numbers at the end

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


...

Error processing batch 9 for Key Terms: Failed after 5 retries: 429 Resource has been exhausted (e.g. check quota).
Processing batch 10 for: Extract the key types of problems discussed in the following slides and provide examples if possible. Your problem types should be specific to this lecture, but also make sense as a general problem in the context of Linear Programming. Respond in the following format: <problem type>: <description>. Do not include any other text, like numbering, intermediate references, or general summaries before/after the problem type. Do not focus on generating Key Terms or Algorithm Solutions since this will be done in another section. If you are citing a slide, include the slide number at the end of the term. Here is an example: 'Verifying Optimality: A method for verifying the optimality of a solution is presented, involving checking the objective function value and the feasibility of the dual solution.<SLIDE 13>'. If citing multiple slides, include the slide

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


Error processing batch 9 for Problem Types: Failed after 5 retries: 429 Resource has been exhausted (e.g. check quota).
Processing batch 10 for: Extract the key algorithms to solve the problems from the following slides, and explain their meaning briefly. Your algorithms should be specific to this lecture, but also make sense as a general algorithm solution in the context of Linear Programming. Using formulas is encouraged. Respond in the following format: <algorithm>: <formula and/or explanation>. Do not include any other text, like numbering, intermediate references, or general summaries before/after the algorithm. Do not focus on generating Key Terms or Problem Types since this will be done in another section. If you are citing a slide, include the slide number at the end of the term. Here is an example: 'Strong Duality: This theorem states that the optimal objective function values of the primal and dual problems are equal.<SLIDE 2>'. If citing multiple slides, include the slide nu

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


Terms:  ['branch and bound', 'gomory cuts']
Pruning term: gomory cuts
Processing batch 16 for: Extract the key terms from the following slides and provide a clear and concise definition for each one. Your key terms should be specific to this lecture, but also make sense as a general topic in the context of Linear Programming. Respond in the following format: <term>: <definition>. Do not include any other text, like numbering, intermediate references, or general summaries before/after the term. Do not focus on generating Problem Types or Algorithm Solutions since this will be done in another section. If you are citing a slide, include the slide number at the end of the term. Here is an example: 'Normal Equation: A closed-form solution for the least squares problem in linear regression. It's always solvable, even if the original system of equations is not.<SLIDE 12>'. If citing multiple slides, include the slide numbers at the end of the definition. Here is another example: 'Support Vect